In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os

# эта часть взята из baseline



# сид рандома 
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(68)
TRACK = "team"
TRAIN_DAYS = 14
MAX_TRAIN_ROWS = 1_500_000
RIDGE_ALPHA = 4.0
RANDOM_STATE = 42

TRACK_CONFIG = {
    "solo": {
        "train_path": "train_solo_track.parquet",
        "test_path": "test_solo_track.parquet",
        "target_col": "target_1h",
        "forecast_points": 8,
    },
    "team": {
        "train_path": "train_team_track.parquet",
        "test_path": "test_team_track.parquet",
        "target_col": "target_2h",
        "forecast_points": 10,
    },
}

CONFIG = TRACK_CONFIG[TRACK]
TARGET_COL = CONFIG["target_col"]
FORECAST_POINTS = CONFIG["forecast_points"]
FUTURE_TARGET_COLS = [f"target_step_{step}" for step in range(1, FORECAST_POINTS + 1)]

test_df = pd.read_parquet("test_team_track.parquet")
train_df = pd.read_parquet("train_team_track.parquet")

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print("track:", TRACK)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

from tqdm import tqdm

### ниже модель, создание датасета фич экстракшн и создание посылки для теста


In [ ]:
# feature extraction и создание датасета использованы среднее,стандартное отклонение,время и дни недели через тригонометрические функции,
def add_features(df):
    df = df.copy()
    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    cols = [c for c in df.columns if "status" in c]
    print(cols)
    df["nulls"]=(df[cols]==0).astype("float32").sum(axis=1)
    print(df["nulls"].unique())
    print((df[cols]).head(5))
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    
    
    g = df.groupby("route_id")[TARGET_COL]

    for lag in [1, 2, 3, 6, 12, 24, 48,96]:
        df[f"lag_{lag}"] = g.shift(lag)
    df["diff_1"] = g.diff(1)
    df["diff_24"] = g.diff(24)
    
    for w in [5, 10, 20, 50,100]:
        df[f"roll_mean_{w}"] = (
            g.rolling(w).mean().reset_index(level=0, drop=True)
        )
        df[f"roll_std_{w}"] = (
            g.rolling(w).std().reset_index(level=0, drop=True)
        )
    
    return df



train_df = add_features(train_df)
train_df = train_df.dropna().reset_index(drop=True)

feature_cols = [c for c in train_df.columns if c not in ["timestamp", "route_id"]]
# используем скейлер, дает прирост для нейросетей.
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols]).astype("float32")
SEQ_LEN = 240
HORIZON = FORECAST_POINTS
unique_timestamps = np.sort(train_df['timestamp'].unique())
# 95 процентов на трейн 5 на валидацию сплит по времени
split_idx = int(len(unique_timestamps) * 0.95)
split_time = unique_timestamps[split_idx]
split_time_val = unique_timestamps[split_idx - SEQ_LEN]
print(f"Граница раздела по времени: {split_time}")
del unique_timestamps
train_df_split = train_df[train_df['timestamp'] <= split_time].copy()
val_df_split = train_df[train_df['timestamp'] > split_time_val].copy()
print(train_df.shape)
print(f"Train size: {len(train_df_split)}, Val size: {len(val_df_split)}")





print(train_df_split.shape)
print(val_df_split.shape)


import torch
from torch.utils.data import Dataset

# датасет 
class WindowDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len, horizon):
        self.seq_len = seq_len
        self.horizon = horizon
        self.feature_cols = feature_cols
        self.target_idx = feature_cols.index(TARGET_COL)

        self.groups = []
        self.indices = []
        for g_idx, (_, group) in enumerate(df.groupby("route_id")):
            data = group[feature_cols].values.astype("float32")

            if len(data) > seq_len + horizon:
                self.groups.append(data)

                for i in range(len(data) - seq_len - horizon):
                    self.indices.append((len(self.groups) - 1, i))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        g_idx, i = self.indices[idx]
        data = self.groups[g_idx]

        x = data[i:i + self.seq_len]
        y = data[i + self.seq_len:i + self.seq_len + self.horizon, self.target_idx]

        return (
            torch.from_numpy(x),
            torch.from_numpy(y)
        )


from torch.utils.data import DataLoader
# освобождение памяти, создание лоадера 
train_dataset = WindowDataset(train_df_split, feature_cols, SEQ_LEN, HORIZON)
del train_df_split
val_dataset = WindowDataset(val_df_split, feature_cols, SEQ_LEN, HORIZON)
del val_df_split
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

In [ ]:
# классическая tcn модель глубина 8 hiden 32, увелечение глубины ширины или Hiden size прироста не дали небольшое подобие self atention
class Block(nn.Module):
    def __init__(self, ch, dilation, dropout):
        super().__init__()
        self.conv = nn.Conv1d(
            ch, ch,
            kernel_size=3,
            padding=dilation,
            dilation=dilation
        )
        self.norm = nn.GroupNorm(1, ch)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        res = x

        x = self.conv(x)
        x = self.norm(x)
        x = F.gelu(x)
        x = self.dropout(x)

        return x + res


class TCN(nn.Module):
    def __init__(self, in_channels, hidden=32, depth=5, dropout=0.18):
        super().__init__()

        self.input_proj = nn.Conv1d(in_channels, hidden, kernel_size=1)

        self.blocks=nn.Sequential()

        for i in range(1):
            gay=Block(hidden, dilation=2 ** i, dropout=dropout)
            self.blocks.append(gay)
        for i in range(7):
            gay = Block(hidden, dilation=2 ** (i+1), dropout=dropout)
            self.blocks.append(gay)

        self.attn = nn.Linear(hidden, 1)

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 10)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.input_proj(x)
        x = self.blocks(x)

        x = x.transpose(1, 2)

        w = torch.softmax(self.attn(x), dim=1)
        x = (x * w).sum(dim=1)

        return self.head(x)


device = "cuda" if torch.cuda.is_available() else "cpu"

model = TCN(in_channels=37).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


loss_fn = nn.SmoothL1Loss()

In [ ]:
# метрика соревнования
class WapePlusRbias:
    @property
    def name(self) -> str:
        return "wape_plus_rbias"

    def calculate(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        wape = (np.abs(y_pred - y_true)).sum() / y_true.sum()
        rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
        return wape + rbias


metric = WapePlusRbias()

EPOCHS = 80
# хороший скедулер 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.7,
    patience=4,
    verbose=True
)

# трейн
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch} [TRAIN]")

    for x, y in train_bar:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })
    model.eval()
    val_loss = 0
    all_preds = []
    all_targets = []
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch} [VAL]")
    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device)
            y = y.to(device)
            pred = model(x)
            loss = loss_fn(pred, y)

            val_loss += loss.item()

            all_preds.append(pred.cpu().numpy())
            all_targets.append(y.cpu().numpy())
    scheduler.step(val_loss)

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    preds_flat = all_preds.reshape(-1)
    targets_flat = all_targets.reshape(-1)

    # обратное преобразование по скейлеру
    tmp_pred = np.zeros((len(preds_flat), len(feature_cols)), dtype="float32")
    tmp_true = np.zeros_like(tmp_pred)

    target_idx = feature_cols.index(TARGET_COL)

    tmp_pred[:, target_idx] = preds_flat
    tmp_true[:, target_idx] = targets_flat

    tmp_pred = scaler.inverse_transform(tmp_pred)
    tmp_true = scaler.inverse_transform(tmp_true)

    preds_real = tmp_pred[:, target_idx]
    targets_real = tmp_true[:, target_idx]
    # округление до целого т.к. в соревнование только целый объем
    preds_real=np.round(preds_real)

    val_metric = metric.calculate(targets_real, preds_real)

    print(
        f"\nEpoch {epoch} | "
        f"train_loss={train_loss / len(train_loader):.4f} | "
        f"val_loss={val_loss / len(val_loader):.4f} | "
        f"val_wape+rbias={val_metric:.4f}\n"
    )
    file_path = os.path.join("models", f"len_240_rndm_state_68_epoch_{epoch}_deep=8_hidden=32_bestmodel_val_metric{val_metric}.pt")
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }, file_path)

In [ ]:
# лучшая модель по посылке

path="models/rndm_state_42_epoch_21_deep=8_hidden=32_bestmodel_dif_lern_dif_lr_val_metric0.24610000848770142.pt"
ckpt = torch.load(path, map_location=device)
model.load_state_dict(ckpt["model"])
optimizer.load_state_dict(ckpt["optimizer"])


In [ ]:
# создание посылки для теста
model.eval()
target_idx = feature_cols.index(TARGET_COL)
all_preds = []
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

for route_id, test_group in test_df.groupby("route_id"):
    train_group = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(train_group) < SEQ_LEN:
        preds_real = [0] * len(test_group)
    else:
        window = train_group[feature_cols].values[-SEQ_LEN:].astype("float32")

        x = torch.from_numpy(window).unsqueeze(0).to(device)
        
        with torch.no_grad():
            preds = model(x).cpu().numpy()[0]  

        tmp = np.zeros((len(preds), len(feature_cols)), dtype="float32")
        tmp[:, target_idx] = preds
        tmp = scaler.inverse_transform(tmp)
        
        preds_real = tmp[:, target_idx]
    
    all_preds.extend(preds_real[:len(test_group)])
submission = test_df.copy()
submission["y_pred"] = all_preds
submission = submission[["id", "y_pred"]]
submission.to_csv("submission_final.csv", index=False)

### Ниже создание датасета для проверки предсказаний по валидации, создание предсказаний с шагом в 5 часов для паказа на графиках в презентации


In [ ]:
# тоже самое только в лоадер добавляется информация о времени
def add_features(df):
    df = df.copy()
    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    cols = [c for c in df.columns if "status" in c]
    print(cols)
    df["nulls"]=(df[cols]==0).astype("float32").sum(axis=1)
    print(df["nulls"].unique())
    print((df[cols]).head(5))
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    
    
    g = df.groupby("route_id")[TARGET_COL]

    for lag in [1, 2, 3, 6, 12, 24, 48,96]:
        df[f"lag_{lag}"] = g.shift(lag)
    df["diff_1"] = g.diff(1)
    df["diff_24"] = g.diff(24)
    
    for w in [5, 10, 20, 50,100]:
        df[f"roll_mean_{w}"] = (
            g.rolling(w).mean().reset_index(level=0, drop=True)
        )
        df[f"roll_std_{w}"] = (
            g.rolling(w).std().reset_index(level=0, drop=True)
        )
    
    return df



train_df = add_features(train_df)
train_df = train_df.dropna().reset_index(drop=True)

feature_cols = [c for c in train_df.columns if c not in ["timestamp", "route_id"]]

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols]).astype("float32")
SEQ_LEN = 240
HORIZON = FORECAST_POINTS
unique_timestamps = np.sort(train_df['timestamp'].unique())
split_idx = int(len(unique_timestamps) * 0.95)
split_time = unique_timestamps[split_idx]
split_time_val = unique_timestamps[split_idx - SEQ_LEN]
print(f"Граница раздела по времени: {split_time}")
del unique_timestamps
train_df_split = train_df[train_df['timestamp'] <= split_time].copy()
val_df_split = train_df[train_df['timestamp'] > split_time_val].copy()
print(train_df.shape)
print(f"Train size: {len(train_df_split)}, Val size: {len(val_df_split)}")





print(train_df_split.shape)
print(val_df_split.shape)


import torch
from torch.utils.data import Dataset


class WindowDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len, horizon):
        self.seq_len = seq_len
        self.horizon = horizon
        self.feature_cols = feature_cols
        self.target_idx = feature_cols.index(TARGET_COL)
        self.meta_groups = []

        self.groups = []
        self.indices = []
        for g_idx, (route_id, group) in enumerate(df.groupby("route_id")):
            group = group.sort_values("timestamp")
            data = group[feature_cols].values.astype("float32")
            meta = group[["timestamp", "route_id", "office_from_id"]].reset_index(drop=True)
            meta["timestamp"] = meta["timestamp"].astype("int64")
            if len(data) > seq_len + horizon:
                self.groups.append(data)
                self.meta_groups.append(meta)
                for i in range(len(data) - seq_len - horizon):
                    self.indices.append((len(self.groups) - 1, i))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        g_idx, i = self.indices[idx]
        data = self.groups[g_idx]
        meta = self.meta_groups[g_idx]
        x = data[i:i + self.seq_len]
        y = data[i + self.seq_len:i + self.seq_len + self.horizon, self.target_idx]
        meta_slice = meta.iloc[i + self.seq_len : i + self.seq_len + self.horizon]
        return (
            torch.from_numpy(x),
            torch.from_numpy(y),
            {
                "timestamp": meta_slice["timestamp"].tolist(),
                "route_id": meta_slice["route_id"].values,
                "office_from_id": meta_slice["office_from_id"].values
            }
        )


from torch.utils.data import DataLoader

train_dataset = WindowDataset(train_df_split, feature_cols, SEQ_LEN, HORIZON)
del train_df_split
val_dataset = WindowDataset(val_df_split, feature_cols, SEQ_LEN, HORIZON)
del val_df_split
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

In [ ]:
val_records = []
STEP = 10 #- шаг предсказания как раз 5 часов для 


cur_route=-1
cur_step=0
with torch.no_grad():
    for batch_idx, (x, y, meta) in enumerate(val_loader):
        x = x.to(device)
        y = y.to(device)

        pred = model(x)

        preds_np = pred.cpu().numpy()
        targets_np = y.cpu().numpy()

        B, H = preds_np.shape
# возможно лучше было усреднить для качества, но для проверки именно применимости лучше оставить так как есть

        for i in range(B):
            if cur_route!=meta["route_id"][i][0]:
                cur_route=meta["route_id"][i][0]
                cur_step=0
            if cur_step%STEP==0:
                for t in range(H):
                    val_records.append({
                        "timestamp": meta["timestamp"][t][i],
                        "route_id": meta["route_id"][i][t],
                        "office_from_id": meta["office_from_id"][i][t],
                        "y_true": targets_np[i][t],
                        "y_pred": preds_np[i][t]
                    })
            cur_step+=1
val_df = pd.DataFrame(val_records)
val_df["timestamp"] = val_df["timestamp"].apply(lambda x: x.item() if hasattr(x, "item") else x)
val_df["timestamp"] = pd.to_datetime(val_df["timestamp"])

tmp_pred = np.zeros((len(val_df), len(feature_cols)), dtype="float32")
tmp_true = np.zeros_like(tmp_pred)

target_idx = feature_cols.index(TARGET_COL)

tmp_pred[:, target_idx] = val_df["y_pred"].values
tmp_true[:, target_idx] = val_df["y_true"].values

tmp_pred = scaler.inverse_transform(tmp_pred)
tmp_true = scaler.inverse_transform(tmp_true)

tmp_office = np.zeros((len(val_df), len(feature_cols)), dtype="float32")
office_idx = feature_cols.index("office_from_id")

tmp_office[:, office_idx] = val_df["office_from_id"].values
tmp_office = scaler.inverse_transform(tmp_office)

val_df["office_from_id"] = np.round(tmp_office[:, office_idx]).astype(int)

val_df["route_id"] = val_df["route_id"].astype(int)



val_df["y_pred_real"] = np.round(tmp_pred[:, target_idx])
val_df["y_true_real"] = tmp_true[:, target_idx]
val_df.to_csv("val_predictions.csv", index=False)
